In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Helper to format unique lists with optional limit
def format_unique_list(values, limit=None):
    uniq = sorted(pd.unique(values.astype(str)))
    if limit is None or len(uniq) <= limit:
        return ", ".join(uniq)
    else:
        return ", ".join(uniq[:limit]) + f", ...(+{len(uniq)-limit} more)"

# Build compact summary per file
summaries = []

# metric.csv summary
summaries.append({
    "file": "metric.csv",
    "row_count": len(metric_df),
    "min_timestamp_utc": metric_df['timestamp'].min().isoformat() if not metric_df.empty else None,
    "max_timestamp_utc": metric_df['timestamp'].max().isoformat() if not metric_df.empty else None,
    "unique_cmdb_ids": format_unique_list(metric_df['cmdb_id']),  # all cmdb ids
    "unique_kpi_names_top20": format_unique_list(metric_df['kpi_name'], limit=20)
})

# trace.csv summary
summaries.append({
    "file": "trace.csv",
    "row_count": len(trace_df),
    "min_timestamp_utc": trace_df['timestamp'].min().isoformat() if not trace_df.empty else None,
    "max_timestamp_utc": trace_df['timestamp'].max().isoformat() if not trace_df.empty else None,
    "unique_cmdb_ids": format_unique_list(trace_df['cmdb_id']),
    "unique_trace_names_top20": format_unique_list(trace_df['trace_name'], limit=20)
})

# log.csv summary
summaries.append({
    "file": "log.csv",
    "row_count": len(log_df),
    "min_timestamp_utc": log_df['timestamp'].min().isoformat() if not log_df.empty else None,
    "max_timestamp_utc": log_df['timestamp'].max().isoformat() if not log_df.empty else None,
    "unique_cmdb_ids": format_unique_list(log_df['cmdb_id']),
    "unique_log_names_top20": format_unique_list(log_df['log_name'], limit=20)
})

# error_logs.csv summary (no KPI/trace/log name list)
summaries.append({
    "file": "error_logs.csv",
    "row_count": len(error_df),
    "min_timestamp_utc": error_df['timestamp'].min().isoformat() if not error_df.empty else None,
    "max_timestamp_utc": error_df['timestamp'].max().isoformat() if not error_df.empty else None,
    "unique_cmdb_ids": format_unique_list(error_df['cmdb_id']),
    "unique_names_top20": ""  # intentionally empty / skipped as requested
})

# Create a compact DataFrame summary
summary_df = pd.DataFrame(summaries)[[
    "file", "row_count", "min_timestamp_utc", "max_timestamp_utc", "unique_cmdb_ids",
    "unique_kpi_names_top20", "unique_trace_names_top20", "unique_log_names_top20", "unique_names_top20"
]].fillna("")

# Keep columns compact: some will be empty for rows where not applicable
summary_df

```
Out[1]:
```
Here’s a concise plain-English summary of the execution results:

- metric.csv
  - Rows: 1775
  - Time range (UTC): 2024-01-17T12:06:00+00:00 → 2024-01-17T12:30:00+00:00
  - CMDB IDs (examples): adservice, cartservice, checkoutservice, currencyservice, … (all unique IDs listed in file)
  - KPI names (top examples): cpu, diskio, error, latency-50, latency-90, memory, … (up to 20 shown)

- trace.csv
  - Rows: 3224
  - Time range (UTC): 2024-01-17T12:06:00+00:00 → 2024-01-17T12:30:00+00:00
  - CMDB IDs (examples): checkoutservice, currencyservice, emailservice, … (all unique IDs listed)
  - Trace names (top examples): trace.from_checkoutservice.duration_mean, trace.self.duration_mean, … (up to 20 shown)

- log.csv
  - Rows: 460
  - Time range (UTC): 2024-01-17T12:06:00+00:00 → 2024-01-17T12:30:00+00:00
  - CMDB IDs (examples): adservice, cartservice, checkoutservice, currencyservice, … (all unique IDs listed)
  - Log names: log.error_count, log.row_count (listed)

- error_logs.csv
  - Rows: 12
  - Time range (UTC): 2024-01-17T12:18:49+00:00 → 2024-01-17T12:29:59+00:00
  - CMDB IDs: frontend (only)

If you want, I can expand any of these lists fully (up to 20 entries), show counts per service, or extract specific KPI/trace/log name occurrences.

The original code execution output of IPython Kernel is also provided below for reference:

file  row_count          min_timestamp_utc          max_timestamp_utc                                    unique_cmdb_ids                             unique_kpi_names_top20                           unique_trace_names_top20          unique_log_names_top20 unique_names_top20
0      metric.csv       1775  2024-01-17T12:06:00+00:00  2024-01-17T12:30:00+00:00  adservice, cartservice, checkoutservice, curre...  cpu, diskio, error, latency-50, latency-90, me...                                                                                                      
1       trace.csv       3224  2024-01-17T12:06:00+00:00  2024-01-17T12:30:00+00:00  checkoutservice, currencyservice, emailservice...                                                     trace.from_checkoutservice.duration_mean, trac...                                                   
2         log.csv        460  2024-01-17T12:06:00+00:00  2024-01-17T12:30:00+00:00  adservice, cartservice, checkoutservice, curre...                                                                                                        log.error_count, log.row_count                   
3  error_logs.csv         12  2024-01-17T12:18:49+00:00  2024-01-17T12:29:59+00:00                                           frontend```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Re-load metric.csv to ensure fresh state (timestamps will be parsed to UTC)
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 12:03:28", utc=True)
end = pd.to_datetime("2024-01-17 12:33:28", utc=True)

def find_anomalies_with_quantile(metric_df, q):
    # 1) compute global threshold per (cmdb_id, kpi_name) using entire file
    thresh = (
        metric_df
        .groupby(['cmdb_id', 'kpi_name'])['value']
        .quantile(q)
        .reset_index()
        .rename(columns={'value': 'threshold'})
    )
    # 2) filter rows to the incident window
    window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()
    if window.empty:
        return pd.DataFrame(), thresh  # no data in window

    # 3) attach threshold to window rows
    window = window.merge(thresh, on=['cmdb_id', 'kpi_name'], how='left')

    # 4) mark anomalies where value >= threshold (treat equality as anomaly)
    window['is_anom'] = window['value'] >= window['threshold']

    # 5) aggregate per pair
    totals = window.groupby(['cmdb_id', 'kpi_name']).agg(
        total_points_in_window=('value','size'),
        max_value_in_window=('value','max')
    ).reset_index()

    anom_agg = (
        window[window['is_anom']]
        .groupby(['cmdb_id', 'kpi_name'])
        .agg(
            anomaly_count_in_window=('is_anom','sum'),
            earliest_anomaly_timestamp_utc=('timestamp','min')
        )
        .reset_index()
    )

    # merge totals and anomaly aggregates
    merged = totals.merge(anom_agg, on=['cmdb_id','kpi_name'], how='left')
    # fill NaN anomaly counts with 0
    merged['anomaly_count_in_window'] = merged['anomaly_count_in_window'].fillna(0).astype(int)
    # compute fraction
    merged['anomaly_fraction_in_window'] = merged['anomaly_count_in_window'] / merged['total_points_in_window']

    # attach threshold values (from thresh)
    merged = merged.merge(thresh, on=['cmdb_id','kpi_name'], how='left')

    # keep only pairs with at least one anomaly
    candidates = merged[merged['anomaly_count_in_window'] > 0].copy()
    if candidates.empty:
        return pd.DataFrame(), thresh

    # format earliest timestamp to ISO strings
    candidates['earliest_anomaly_timestamp_utc'] = candidates['earliest_anomaly_timestamp_utc'].apply(
        lambda ts: ts.isoformat() if pd.notnull(ts) else None
    )

    # rename columns, include threshold label
    candidates = candidates.rename(columns={
        'threshold': f'P{int(q*100)}_threshold',
        'max_value_in_window': 'max_value_in_window'
    })

    # sort by anomaly_count desc, anomaly_fraction desc, earliest timestamp asc
    candidates = candidates.sort_values(
        by=['anomaly_count_in_window', 'anomaly_fraction_in_window', 'earliest_anomaly_timestamp_utc'],
        ascending=[False, False, True]
    )

    # select requested columns and limit to 20 rows
    out_cols = [
        'cmdb_id', 'kpi_name', f'P{int(q*100)}_threshold',
        'total_points_in_window', 'anomaly_count_in_window', 'anomaly_fraction_in_window',
        'earliest_anomaly_timestamp_utc', 'max_value_in_window'
    ]
    return candidates[out_cols].head(20), thresh

# Try P95 first
result_df, p_thresh_95 = find_anomalies_with_quantile(metric_df, 0.95)

if result_df.empty:
    # Retry with P90 if no anomalies found at P95
    result_df, p_thresh_90 = find_anomalies_with_quantile(metric_df, 0.90)
    if not result_df.empty:
        # add label column to indicate threshold used
        result_df['threshold_used'] = 'P90'
else:
    result_df['threshold_used'] = 'P95'

# Final compact result (may be empty DataFrame if no anomalies at P90 either)
result_df

```
Out[2]:
```
Summary (plain English):

- Threshold used: P95 (global per (cmdb_id, kpi_name) computed across the whole metric.csv).
- Candidate pairs with anomalies in the incident window: 20 pairs (table limited to top 20).
- Many services show socket-related anomalies that affect every sampled point in the window (anomaly_fraction = 1.00). These fully-anomalous socket pairs include:
  - adservice.socket, cartservice.socket, checkoutservice.socket, currencyservice.socket, paymentservice.socket, redis.socket, shippingservice.socket, recommendationservice.socket
  - plus emailservice.socket and several services with error/diskio metrics marked anomalous (emailservice.diskio, frontend.error, frontend-external.error, recommendationservice.diskio).
- Partial/anomalous-fraction pairs:
  - productcatalogservice.socket: 12/25 points anomalous (0.48), earliest anomaly 2024-01-17T12:19:00+00:00
  - frontend.socket: 4/25 (0.16), earliest 2024-01-17T12:27:00+00:00
  - checkoutservice.cpu and checkoutservice.workload: 2/25 each (0.08), earliest 2024-01-17T12:06:00+00:00
  - currencyservice.cpu and currencyservice.mem: 2/25 each (0.08), earliest 2024-01-17T12:06:00+00:00
  - emailservice.cpu: 2/25 (0.08)
- Timing: most earliest anomalies occur at 2024-01-17T12:06:00+00:00 (shortly after the window start).
- Interpretation: the dominant anomaly signal is elevated "socket" metrics across many services (many with anomaly_fraction = 1.0), indicating socket-related issues are the primary candidate root cause in this window. Other smaller signals (CPU, memory, workload, productcatalogservice socket, frontend socket) appear but are less pervasive.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  kpi_name  P95_threshold  total_points_in_window  anomaly_count_in_window  anomaly_fraction_in_window earliest_anomaly_timestamp_utc  max_value_in_window threshold_used
6               adservice    socket   4.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         4.000000e+00            P95
12            cartservice    socket   4.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         4.000000e+00            P95
18        checkoutservice    socket   9.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         9.000000e+00            P95
24        currencyservice    socket   4.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         4.000000e+00            P95
27           emailservice    diskio   0.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         0.000000e+00            P95
31           emailservice    socket   3.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         3.000000e+00            P95
34               frontend     error   0.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         0.000000e+00            P95
40      frontend-external     error   0.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         0.000000e+00            P95
46         paymentservice    socket   3.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         3.000000e+00            P95
55  recommendationservice    diskio   0.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         0.000000e+00            P95
59  recommendationservice    socket   4.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         4.366667e+00            P95
64                  redis    socket   4.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         4.000000e+00            P95
69        shippingservice    socket   2.000000e+00                      25                       25                        1.00      2024-01-17T12:06:00+00:00         2.000000e+00            P95
52  productcatalogservice    socket   7.000000e+00                      25                       12                        0.48      2024-01-17T12:19:00+00:00         7.000000e+00            P95
38               frontend    socket   4.400000e+01                      25                        4                        0.16      2024-01-17T12:27:00+00:00         4.400000e+01            P95
14        checkoutservice       cpu   5.864132e-01                      25                        2                        0.08      2024-01-17T12:06:00+00:00         6.135942e-01            P95
19        checkoutservice  workload   1.238002e+00                      25                        2                        0.08      2024-01-17T12:06:00+00:00         1.440133e+00            P95
20        currencyservice       cpu   3.238882e+01                      25                        2                        0.08      2024-01-17T12:06:00+00:00         3.361200e+01            P95
23        currencyservice       mem   4.514856e+07                      25                        2                        0.08      2024-01-17T12:06:00+00:00         4.527514e+07            P95
26           emailservice       cpu   3.288387e-01                      25                        2                        0.08      2024-01-17T12:06:00+00:00         3.420118e-01            P95```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse/ensure trace_df is loaded and timestamps parsed as UTC
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Parameters
services = [
    "checkoutservice", "currencyservice", "emailservice",
    "productcatalogservice", "recommendationservice"
]
start = pd.to_datetime("2024-01-17 12:03:28", utc=True)
end = pd.to_datetime("2024-01-17 12:33:28", utc=True)

def anomalies_for_quantile(trace_df, services, q):
    # 1) global threshold per (cmdb_id, trace_name)
    thresh = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'])['value']
        .quantile(q)
        .reset_index()
        .rename(columns={'value': 'threshold'})
    )

    # 2) filter to services and incident window
    window = trace_df[
        (trace_df['cmdb_id'].isin(services)) &
        (trace_df['timestamp'] >= start) &
        (trace_df['timestamp'] <= end)
    ].copy()

    if window.empty:
        return pd.DataFrame(), thresh

    # 3) attach threshold
    window = window.merge(thresh, on=['cmdb_id', 'trace_name'], how='left')

    # 4) mark anomalies where value >= threshold
    window['is_anom'] = window['value'] >= window['threshold']

    # 5) aggregate per pair
    totals = window.groupby(['cmdb_id', 'trace_name']).agg(
        total_points_in_window=('value', 'size'),
        max_value_in_window=('value','max')
    ).reset_index()

    anom_agg = (
        window[window['is_anom']]
        .groupby(['cmdb_id', 'trace_name'])
        .agg(
            anomaly_count_in_window=('is_anom','sum'),
            earliest_anomaly_timestamp_utc=('timestamp','min')
        )
        .reset_index()
    )

    merged = totals.merge(anom_agg, on=['cmdb_id','trace_name'], how='left')
    merged['anomaly_count_in_window'] = merged['anomaly_count_in_window'].fillna(0).astype(int)
    merged['anomaly_fraction_in_window'] = merged['anomaly_count_in_window'] / merged['total_points_in_window']

    # attach threshold
    merged = merged.merge(thresh, on=['cmdb_id','trace_name'], how='left')

    # keep only pairs with at least one anomaly
    candidates = merged[merged['anomaly_count_in_window'] > 0].copy()
    if candidates.empty:
        return pd.DataFrame(), thresh

    # format earliest timestamp iso
    candidates['earliest_anomaly_timestamp_utc'] = candidates['earliest_anomaly_timestamp_utc'].apply(
        lambda ts: ts.isoformat() if pd.notnull(ts) else None
    )

    # rename threshold column to indicate quantile
    candidates = candidates.rename(columns={'threshold': f'P{int(q*100)}_threshold'})

    # sort and limit
    candidates = candidates.sort_values(
        by=['anomaly_count_in_window', 'anomaly_fraction_in_window', 'earliest_anomaly_timestamp_utc'],
        ascending=[False, False, True]
    )

    out_cols = [
        'cmdb_id', 'trace_name', f'P{int(q*100)}_threshold',
        'total_points_in_window', 'anomaly_count_in_window', 'anomaly_fraction_in_window',
        'earliest_anomaly_timestamp_utc', 'max_value_in_window'
    ]
    return candidates[out_cols].head(20), thresh

# Try P95 first
result_df, thresh_95 = anomalies_for_quantile(trace_df, services, 0.95)

if result_df.empty:
    # Retry with P90
    result_df, thresh_90 = anomalies_for_quantile(trace_df, services, 0.90)
    if not result_df.empty:
        result_df['threshold_used'] = 'P90'
else:
    result_df['threshold_used'] = 'P95'

# Final compact result
result_df

```
Out[3]:
```
Summary (plain English):

- Threshold used: global P95 per (cmdb_id, trace_name).
- Services checked: checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice.
- Result: many trace error_rate series triggered anomalies at P95. The table (top 20 shown) lists pairs where anomaly_count equals total_points in the window (anomaly_fraction = 1.0).
- Observations:
  - The P95 threshold for all reported trace error_rate series is 0.0.
  - All points in the incident window met value >= P95 (so anomaly_count = total_points; max_value_in_window = 0.0 for those rows).
  - Earliest anomalies are mostly at 2024-01-17T12:06:00+00:00.
- Note: Because the computed P95 is zero, the >= rule flags values equal to 0.0 as anomalies; this behavior means these results reflect the threshold choice rather than necessarily elevated non-zero error rates. If you want a stricter signal, we can re-run using P90 or require value > threshold.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  P95_threshold  total_points_in_window  anomaly_count_in_window  anomaly_fraction_in_window earliest_anomaly_timestamp_utc  max_value_in_window threshold_used
42        currencyservice                   trace.from_root.error_rate            0.0                      25                       25                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
50           emailservice                   trace.from_root.error_rate            0.0                      23                       23                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
66  productcatalogservice                   trace.from_root.error_rate            0.0                      23                       23                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
78  recommendationservice                   trace.from_root.error_rate            0.0                      23                       23                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
2         checkoutservice        trace.from_checkoutservice.error_rate            0.0                      22                       22                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
10        checkoutservice                   trace.from_root.error_rate            0.0                      22                       22                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
14        checkoutservice          trace.to_checkoutservice.error_rate            0.0                      22                       22                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
74  recommendationservice  trace.from_recommendationservice.error_rate            0.0                      21                       21                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
86  recommendationservice    trace.to_recommendationservice.error_rate            0.0                      21                       21                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
18        checkoutservice          trace.to_currencyservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
22        checkoutservice             trace.to_emailservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
34        currencyservice        trace.from_checkoutservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
38        currencyservice        trace.from_frontendservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
46           emailservice        trace.from_checkoutservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
58  productcatalogservice        trace.from_frontendservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
62  productcatalogservice  trace.from_recommendationservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
70  recommendationservice        trace.from_frontendservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
82  recommendationservice    trace.to_productcatalogservice.error_rate            0.0                      20                       20                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
6         checkoutservice        trace.from_frontendservice.error_rate            0.0                      19                       19                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95
26        checkoutservice           trace.to_paymentservice.error_rate            0.0                      19                       19                         1.0      2024-01-17T12:06:00+00:00                  0.0            P95```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing dataframes if available, otherwise load
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")

# Parse timestamps as UTC datetimes (Unix seconds)
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-01-17 12:03:28", utc=True)
end = pd.to_datetime("2024-01-17 12:33:28", utc=True)

# Filter to window
log_window = log_df[(log_df['timestamp'] >= start) & (log_df['timestamp'] <= end)].copy()
error_window = error_df[(error_df['timestamp'] >= start) & (error_df['timestamp'] <= end)].copy()

# 1) log.csv aggregation grouped by cmdb_id and log_name
grp = (
    log_window
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(total_points_in_window='size', sum_value_in_window='sum', max_value_in_window='max')
    .reset_index()
)

# earliest timestamp in window where value > 0
positive_ts = (
    log_window[log_window['value'] > 0]
    .groupby(['cmdb_id', 'log_name'])['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'earliest_positive_ts_utc'})
)

# merge and format
log_summary_df = grp.merge(positive_ts, on=['cmdb_id', 'log_name'], how='left')
# format timestamp to ISO string
log_summary_df['earliest_positive_ts_utc'] = log_summary_df['earliest_positive_ts_utc'].apply(
    lambda t: t.isoformat() if pd.notnull(t) else None
)

# filter where sum_value_in_window > 0 and sort descending, limit 20
log_summary_df = log_summary_df[log_summary_df['sum_value_in_window'] > 0].sort_values(
    by='sum_value_in_window', ascending=False
).reset_index(drop=True).head(20)

# 2) error_logs.csv summary
error_total_count = len(error_window)

# unique cmdb_id values and counts per cmdb_id
error_cmdb_counts_df = (
    error_window['cmdb_id']
    .value_counts()
    .rename_axis('cmdb_id')
    .reset_index(name='count')
)

# up to 20 sample rows with timestamp_utc, cmdb_id, message
error_samples_df = error_window[['timestamp', 'cmdb_id', 'message']].copy()
error_samples_df['timestamp_utc'] = error_samples_df['timestamp'].apply(lambda t: t.isoformat())
error_samples_df = error_samples_df[['timestamp_utc', 'cmdb_id', 'message']].head(20)

# 3) search error messages for socket/connection-related keywords (case-insensitive)
keywords = ['socket', 'connect', 'connection', 'timeout', 'refused', 'ECONN']
# prepare a dataframe of matches: add a column for message lowercased for search
error_window['message_str'] = error_window['message'].astype(str)

# build keyword counts and samples
records = []
samples_list = []
for kw in keywords:
    mask = error_window['message_str'].str.contains(kw, case=False, na=False)
    matched = error_window[mask].copy()
    count = len(matched)
    records.append({'keyword': kw, 'count': count})
    if count > 0:
        tmp = matched[['timestamp', 'cmdb_id', 'message']].copy()
        tmp['timestamp_utc'] = tmp['timestamp'].apply(lambda t: t.isoformat())
        tmp = tmp[['timestamp_utc', 'cmdb_id', 'message']].head(5)
        tmp['keyword'] = kw
        samples_list.append(tmp)

keyword_counts_df = pd.DataFrame(records)

# concatenate samples for all keywords (will be small, <= 6*5 rows)
if samples_list:
    keyword_samples_df = pd.concat(samples_list, ignore_index=True)[
        ['keyword', 'timestamp_utc', 'cmdb_id', 'message']
    ]
else:
    keyword_samples_df = pd.DataFrame(columns=['keyword', 'timestamp_utc', 'cmdb_id', 'message'])

# Final compact outputs
log_summary_df, error_total_count, error_cmdb_counts_df, error_samples_df, keyword_counts_df, keyword_samples_df

```
Out[4]:
```
summary = (
    "Log summary (window 2024-01-17 12:03:28 → 2024-01-17 12:33:28 UTC):\n"
    "- Top log groups with positive sums (up to 20):\n"
    "  1) frontend / log.row_count: points=25, sum=55751, max=2805, first positive=2024-01-17T12:06:00+00:00\n"
    "  2) currencyservice / log.row_count: points=25, sum=50120, max=2501, first positive=2024-01-17T12:06:00+00:00\n"
    "  3) cartservice / log.row_count: points=25, sum=16093, max=843, first positive=2024-01-17T12:06:00+00:00\n"
    "  4) recommendationservice / log.row_count: points=25, sum=11760, max=612, first positive=2024-01-17T12:06:00+00:00\n"
    "  5) shippingservice / log.row_count: points=25, sum=9418, max=546, first positive=2024-01-17T12:06:00+00:00\n"
    "  6) adservice / log.row_count: points=25, sum=9371, max=465, first positive=2024-01-17T12:06:00+00:00\n"
    "  7) checkoutservice / log.row_count: points=25, sum=1686, max=126, first positive=2024-01-17T12:06:00+00:00\n"
    "  8) paymentservice / log.row_count: points=25, sum=1124, max=84, first positive=2024-01-17T12:06:00+00:00\n"
    "  9) emailservice / log.row_count: points=25, sum=562, max=42, first positive=2024-01-17T12:06:00+00:00\n"
    " 10) redis / log.row_count: points=5, sum=25, max=5, first positive=2024-01-17T12:10:00+00:00\n"
    " 11) frontend / log.error_count: points=25, sum=12, max=3, first positive=2024-01-17T12:18:00+00:00\n\n"
    "Error logs summary (window 2024-01-17 12:03:28 → 2024-01-17 12:33:28 UTC):\n"
    f"- Total error log rows in window: 12\n"
    "- Unique cmdb_id counts:\n"
    "  - frontend: 12\n\n"
    "- Sample error rows (up to 5):\n"
    "  1) 2024-01-17T12:18:49+00:00  frontend  failed to retrieve ads\n"
    "  2) 2024-01-17T12:19:07+00:00  frontend  failed to retrieve ads\n"
    "  3) 2024-01-17T12:19:18+00:00  frontend  failed to retrieve ads\n"
    "  4) 2024-01-17T12:20:49+00:00  frontend  failed to retrieve ads\n"
    "  5) 2024-01-17T12:22:07+00:00  frontend  failed to retrieve ads\n\n"
    "Keyword search for socket/connection-related terms in error messages (case-insensitive):\n"
    "- Keywords searched: ['socket','connect','connection','timeout','refused','ECONN']\n"
    "- Matches found: none (all keyword match counts = 0).\n\n"
    "Conclusion:\n"
    "- Logs show heavy log.row_count activity across many services (frontend and currencyservice highest).\n"
    "- Frontend has the only error_logs entries in the window (12 rows), all with message 'failed to retrieve ads'.\n"
    "- No socket/connection-related keywords were found in the error messages in this window.\n"
    "- If you want, next steps: correlate these frontend errors with the socket anomalies seen in metric/trace data, or provide more detailed log messages for deeper parsing."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name  total_points_in_window  sum_value_in_window  max_value_in_window   earliest_positive_ts_utc
0                frontend    log.row_count                      25                55751                 2805  2024-01-17T12:06:00+00:00
1         currencyservice    log.row_count                      25                50120                 2501  2024-01-17T12:06:00+00:00
2             cartservice    log.row_count                      25                16093                  843  2024-01-17T12:06:00+00:00
3   recommendationservice    log.row_count                      25                11760                  612  2024-01-17T12:06:00+00:00
4         shippingservice    log.row_count                      25                 9418                  546  2024-01-17T12:06:00+00:00
5               adservice    log.row_count                      25                 9371                  465  2024-01-17T12:06:00+00:00
6         checkoutservice    log.row_count                      25                 1686                  126  2024-01-17T12:06:00+00:00
7          paymentservice    log.row_count                      25                 1124                   84  2024-01-17T12:06:00+00:00
8            emailservice    log.row_count                      25                  562                   42  2024-01-17T12:06:00+00:00
9                   redis    log.row_count                       5                   25                    5  2024-01-17T12:10:00+00:00
10               frontend  log.error_count                      25                   12                    3  2024-01-17T12:18:00+00:00, 12,     cmdb_id  count
0  frontend     12,                 timestamp_utc   cmdb_id                 message
0   2024-01-17T12:18:49+00:00  frontend  failed to retrieve ads
1   2024-01-17T12:19:07+00:00  frontend  failed to retrieve ads
2   2024-01-17T12:19:18+00:00  frontend  failed to retrieve ads
3   2024-01-17T12:20:49+00:00  frontend  failed to retrieve ads
4   2024-01-17T12:22:07+00:00  frontend  failed to retrieve ads
5   2024-01-17T12:22:07+00:00  frontend  failed to retrieve ads
6   2024-01-17T12:22:46+00:00  frontend  failed to retrieve ads
7   2024-01-17T12:24:01+00:00  frontend  failed to retrieve ads
8   2024-01-17T12:24:36+00:00  frontend  failed to retrieve ads
9   2024-01-17T12:27:11+00:00  frontend  failed to retrieve ads
10  2024-01-17T12:29:51+00:00  frontend  failed to retrieve ads
11  2024-01-17T12:29:59+00:00  frontend  failed to retrieve ads,       keyword  count
0      socket      0
1     connect      0
2  connection      0
3     timeout      0
4     refused      0
5       ECONN      0, Empty DataFrame
Columns: [keyword, timestamp_utc, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing trace_df if available, otherwise load and parse timestamps
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Parameters
services = [
    "checkoutservice", "currencyservice", "emailservice",
    "productcatalogservice", "recommendationservice"
]
start = pd.to_datetime("2024-01-17 12:03:28", utc=True)
end = pd.to_datetime("2024-01-17 12:33:28", utc=True)

# Helper to run analysis for a given quantile q (e.g., 0.95 or 0.90)
def analyze_duration_traces(trace_df, services, start, end, q):
    # 1) compute global threshold per (cmdb_id, trace_name) using entire trace_df
    thresh = (
        trace_df
        .groupby(['cmdb_id', 'trace_name'])['value']
        .quantile(q)
        .reset_index()
        .rename(columns={'value': 'threshold'})
    )

    # 2) filter to services and incident window
    window = trace_df[
        (trace_df['cmdb_id'].isin(services)) &
        (trace_df['timestamp'] >= start) &
        (trace_df['timestamp'] <= end)
    ].copy()

    if window.empty:
        return pd.DataFrame(), thresh

    # 3) restrict trace_name to those containing 'duration' or 'latency' or 'duration_mean' (case-insensitive)
    name_mask = window['trace_name'].str.contains(r'duration|latency|duration_mean', case=False, na=False)
    # exclude error_rate traces explicitly
    name_mask &= ~window['trace_name'].str.contains(r'error_rate', case=False, na=False)
    window = window[name_mask].copy()
    if window.empty:
        return pd.DataFrame(), thresh

    # 4) attach threshold values (computed globally)
    window = window.merge(thresh, on=['cmdb_id', 'trace_name'], how='left')

    # drop rows where threshold is missing (no global series to compute quantile)
    window = window[~window['threshold'].isna()].copy()
    if window.empty:
        return pd.DataFrame(), thresh

    # mark anomalies where value >= threshold (treat equality as anomaly)
    window['is_anom'] = window['value'] >= window['threshold']

    # aggregate per (cmdb_id, trace_name)
    totals = window.groupby(['cmdb_id', 'trace_name']).agg(
        total_points_in_window=('value', 'size'),
        max_value_in_window=('value', 'max')
    ).reset_index()

    anom_agg = (
        window[window['is_anom']]
        .groupby(['cmdb_id', 'trace_name'])
        .agg(
            anomaly_count_in_window=('is_anom','sum'),
            earliest_anomaly_timestamp_utc=('timestamp','min')
        )
        .reset_index()
    )

    merged = totals.merge(anom_agg, on=['cmdb_id','trace_name'], how='left')
    merged['anomaly_count_in_window'] = merged['anomaly_count_in_window'].fillna(0).astype(int)
    merged['anomaly_fraction_in_window'] = merged['anomaly_count_in_window'] / merged['total_points_in_window']

    # attach threshold
    merged = merged.merge(thresh, on=['cmdb_id','trace_name'], how='left')

    # format earliest timestamp to ISO
    merged['earliest_anomaly_timestamp_utc'] = merged['earliest_anomaly_timestamp_utc'].apply(
        lambda ts: ts.isoformat() if pd.notnull(ts) else None
    )

    # compute severity = max_value_in_window / (threshold if threshold>0 else max_value_in_window+1)
    def compute_sev(row):
        p = row['threshold']
        m = row['max_value_in_window']
        denom = p if (p is not None and p > 0) else (m + 1.0)
        # avoid divide by zero although denom should be >0
        return m / denom if denom != 0 else np.nan

    merged['severity'] = merged.apply(compute_sev, axis=1)

    # keep only pairs that have at least one anomaly (anomaly_count_in_window > 0)
    candidates = merged[merged['anomaly_count_in_window'] > 0].copy()
    if candidates.empty:
        return pd.DataFrame(), thresh

    # prepare output columns and sort by severity desc then anomaly_count desc then earliest timestamp asc
    out_cols = [
        'cmdb_id', 'trace_name', 'threshold', 'total_points_in_window',
        'anomaly_count_in_window', 'anomaly_fraction_in_window',
        'earliest_anomaly_timestamp_utc', 'max_value_in_window', 'severity'
    ]
    candidates = candidates[out_cols].rename(columns={'threshold': f'P{int(q*100)}_threshold'})
    candidates = candidates.sort_values(
        by=['severity', 'anomaly_count_in_window', 'earliest_anomaly_timestamp_utc'],
        ascending=[False, False, True]
    )

    # limit to top 20
    candidates = candidates.head(20).reset_index(drop=True)
    candidates['threshold_used'] = f'P{int(q*100)}'
    return candidates, thresh

# First attempt with P95
result_df, thresh_95 = analyze_duration_traces(trace_df, services, start, end, 0.95)

if result_df.empty:
    # Retry with P90
    result_df, thresh_90 = analyze_duration_traces(trace_df, services, start, end, 0.90)
    if not result_df.empty:
        result_df['threshold_used'] = 'P90'

# Final compact result
result_df

```
Out[5]:
```
Summary (plain English):

- Threshold used: global P95 per (cmdb_id, trace_name). P95 produced anomalies (no fallback to P90 needed).
- Scope: checked duration/latency trace series for checkoutservice, currencyservice, emailservice, productcatalogservice, recommendationservice (excluded error_rate traces).
- Main finding: the strongest duration/latency anomalies are centered on productcatalogservice-related calls and cross-service calls involving productcatalogservice and recommendationservice, with notable spikes around 2024-01-17 ~12:20 UTC and some around ~12:25–12:27 UTC.

Top anomaly candidates (ranked by computed severity, top 5):
1) productcatalogservice — trace.from_recommendationservice.duration_mean
   - P95 = 0.000337, points=20, anomalies=1 (5%), earliest=2024-01-17T12:20:00+00:00, max=0.003645, severity ≈ 10.80
2) recommendationservice — trace.to_productcatalogservice.duration_mean
   - P95 = 0.000337, points=20, anomalies=1 (5%), earliest=2024-01-17T12:20:00+00:00, max=0.003645, severity ≈ 10.80
3) checkoutservice — trace.to_productcatalogservice.duration_mean
   - P95 = 0.000165, points=19, anomalies=1 (~5.3%), earliest=2024-01-17T12:20:00+00:00, max=0.001452, severity ≈ 8.78
4) productcatalogservice — trace.from_checkoutservice.duration_mean
   - P95 = 0.000165, points=19, anomalies=1 (~5.3%), earliest=2024-01-17T12:20:00+00:00, max=0.001452, severity ≈ 8.78
5) checkoutservice — trace.from_root.duration_p95
   - P95 = 0.124766, points=22, anomalies=2 (~9.1%), earliest=2024-01-17T12:25:00+00:00, max=0.417198, severity ≈ 3.34

- Additional notes: Several other duration series show smaller but nonzero anomalies (e.g., productcatalogservice and recommendationservice trace.from_root/trace.from_frontendservice, and some checkoutservice traces). Most anomaly counts are low (1–2 points) but the severity metric (max / P95) highlights a few sharp spikes, especially involving productcatalogservice interactions.

Interpretation / recommended next steps:
- Investigate productcatalogservice around 12:20 UTC (and the calls to it from recommendationservice and checkoutservice). The duration spikes there are the highest-severity signals in the duration traces.
- Correlate these timing spikes with the metric-level socket anomalies and frontend error messages previously observed; this combination suggests productcatalogservice latency (or a downstream dependency of it) may be contributing to the incident.
- Collect logs/host metrics for productcatalogservice and any downstream systems (DB, cache, network) around 12:18–12:27 UTC to find root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  P95_threshold  total_points_in_window  anomaly_count_in_window  anomaly_fraction_in_window earliest_anomaly_timestamp_utc  max_value_in_window   severity threshold_used
0   productcatalogservice  trace.from_recommendationservice.duration_mean       0.000337                      20                        1                    0.050000      2024-01-17T12:20:00+00:00             0.003645  10.802643            P95
1   recommendationservice    trace.to_productcatalogservice.duration_mean       0.000337                      20                        1                    0.050000      2024-01-17T12:20:00+00:00             0.003645  10.802643            P95
2         checkoutservice    trace.to_productcatalogservice.duration_mean       0.000165                      19                        1                    0.052632      2024-01-17T12:20:00+00:00             0.001452   8.777072            P95
3   productcatalogservice        trace.from_checkoutservice.duration_mean       0.000165                      19                        1                    0.052632      2024-01-17T12:20:00+00:00             0.001452   8.777072            P95
4         checkoutservice                    trace.from_root.duration_p95       0.124766                      22                        2                    0.090909      2024-01-17T12:25:00+00:00             0.417198   3.343850            P95
5   productcatalogservice                    trace.from_root.duration_p95       0.176074                      23                        2                    0.086957      2024-01-17T12:18:00+00:00             0.540296   3.068577            P95
6   recommendationservice        trace.from_frontendservice.duration_mean       0.349060                      20                        1                    0.050000      2024-01-17T12:27:00+00:00             0.967058   2.770466            P95
7         checkoutservice                   trace.from_root.duration_mean       0.072782                      22                        2                    0.090909      2024-01-17T12:25:00+00:00             0.154308   2.120128            P95
8   productcatalogservice                   trace.from_root.duration_mean       0.077838                      23                        2                    0.086957      2024-01-17T12:18:00+00:00             0.144592   1.857593            P95
9   recommendationservice                   trace.from_root.duration_mean       0.269617                      23                        2                    0.086957      2024-01-17T12:22:00+00:00             0.450039   1.669177            P95
10           emailservice                   trace.from_root.duration_mean       0.000122                      23                        2                    0.086957      2024-01-17T12:23:00+00:00             0.000183   1.497851            P95
11        checkoutservice           trace.to_paymentservice.duration_mean       0.000325                      19                        1                    0.052632      2024-01-17T12:30:00+00:00             0.000469   1.440753            P95
12        checkoutservice          trace.to_currencyservice.duration_mean       0.001158                      20                        1                    0.050000      2024-01-17T12:13:00+00:00             0.001655   1.429605            P95
13        currencyservice        trace.from_checkoutservice.duration_mean       0.001158                      20                        1                    0.050000      2024-01-17T12:13:00+00:00             0.001655   1.429605            P95
14        currencyservice                   trace.from_root.duration_mean       0.000598                      25                        2                    0.080000      2024-01-17T12:26:00+00:00             0.000814   1.362046            P95
15        checkoutservice        trace.from_checkoutservice.duration_mean       0.059801                      22                        2                    0.090909      2024-01-17T12:21:00+00:00             0.078758   1.317008            P95
16        checkoutservice          trace.to_checkoutservice.duration_mean       0.059801                      22                        2                    0.090909      2024-01-17T12:21:00+00:00             0.078758   1.317008            P95
17        checkoutservice              trace.to_emailservice.duration_p95       0.000577                      20                        1                    0.050000      2024-01-17T12:30:00+00:00             0.000758   1.313337            P95
18           emailservice         trace.from_checkoutservice.duration_p95       0.000577                      20                        1                    0.050000      2024-01-17T12:30:00+00:00             0.000758   1.313337            P95
19        checkoutservice        trace.from_frontendservice.duration_mean       0.599614                      19                        1                    0.052632      2024-01-17T12:21:00+00:00             0.772058   1.287591            P95```
```